# 04 · Three traffic light agents
The traffic light agent proposes the next phase at each junction. Three versions exist, each in its own
file, selected by the controller name:

* **Max-pressure** (`traffix`): green to the phase with the most queued vehicles minus those already
  queued downstream. Greedy, proven to maximise throughput under assumptions (Varaiya, 2013).
* **Predictive / MPC** (`traffix_predictive`): predicts arrivals for the next 60 s (learned arrival model,
  upstream platoons, room downstream) and simulates keep / switch now / switch in 10 s.
* **Reinforcement learning** (`traffix_rl`): Fitted Q-Iteration with gradient-boosted trees, one policy
  shared by all junctions, rewarded for less waiting at its junction and its neighbours.

All three only **propose**: masters and safety rules still decide.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))          # the project root, so `traffix` can be imported
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
from traffix.core import config
from traffix.evaluation import notebook_tools as nt

## 1. The same 15 minutes of morning peak, four controllers

In [ ]:
rows, systems = [], {}
for ctrl in ["actuated", "traffix", "traffix_predictive", "traffix_rl"]:
    r, s, _, narr = nt.short_run("morning_peak", controller=ctrl, minutes=15)
    systems[ctrl] = s
    rows.append({"controller": ctrl, "car wait (s)": r["veh_mean_wait_s"], "pedestrian wait (s)": r["ped_mean_wait_s"],
                 "truncations": r["clearance_truncations"], "unfinished": r["veh_unfinished"]})
pd.DataFrame(rows).set_index("controller").round(1)

## 2. Why each agent switched
Each switch is logged with the agent's own reasoning.

In [ ]:
for ctrl, reason in [("traffix", "MAX_PRESSURE_SWITCH"), ("traffix_predictive", "PREDICTIVE_SWITCH"), ("traffix_rl", "RL_POLICY_SWITCH")]:
    e = next((x for x in systems[ctrl].log.entries if x["reason"] == reason), None)
    if e:
        print(f"{ctrl}: {narr.sentence(e)}\n   details: {e['details']}\n")

## 3. Tuned settings
Max-pressure and the predictive agent have settings tuned by searching over simulations.

In [ ]:
import json
for f in ["traffic_light_params.json", "predictive_params.json"]:
    p = config.path("models", f)
    if os.path.exists(p): print(f, json.load(open(p)))

## 4. Across all scenarios (full experiments)
From `results/results.csv` (3 seeds per scenario), if the experiments have been run.

In [ ]:
res = nt.load_results()
if res is not None:
    sub = res[res.controller.isin(["actuated", "traffix", "traffix_predictive", "traffix_rl"])]
    display(sub.pivot_table(index="scenario", columns="controller", values="veh_mean_wait_s").round(1))
else:
    print("No results yet: run  python scripts/run_experiments.py")